[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 困难：因果自注意力（Causal Self-Attention）

实现 **因果（掩码）自注意力** —— GPT 风格解码器中所使用的注意力机制。

与普通的 softmax 注意力相同，但每个位置**只能关注自身及之前的位置**（不能偷看未来的 token）。

$$\text{scores}_{ij} = \begin{cases} \dfrac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### 函数签名

### 函数签名
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → 输出: (batch, seq, d_v)
```

### 规则
- **禁止**使用 `F.scaled_dot_product_attention`
- 位置 $i$ 只能关注位置 $\le i$
- **允许**使用 `torch.softmax`、`torch.bmm`、`torch.triu`

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

torch.bmm 对三维的矩阵，第一位默认批次，求张量点乘

torch.ones(), 创建全是 1 的张量

torch.triu(input，diagonal) # triangle upper, 上三角取值，其他为零, diagonal 表示对角线偏移量。这是一个关键参数，决定了从哪条线开始切割。diagonal=1 表示从第二个token开始为True

torch.triu(torch.ones(seq_len, seq_len), diagonal=0).bool()<br>
diagonal=0,<br>
[[ True,  True,  True],<br>
[False,  True,  True],<br>
[False, False,  True]]<br>

diagonal=1,<br>
[[False,  True,  True],<br>
[False, False,  True],<br>
[False, False, False]]<br>

torch.tril(input, diagonal) # 下三角，偏移量方向还是向右不是向下<br>
mask = torch.tril(torch.ones(seq_len, seq_len), diagonal=1).bool()<br>
[[ True,  True, False],<br>
[ True,  True,  True],<br>
[ True,  True,  True]]<br>

注意: 
- tensor.masked_fill(input: Tensor, mask: Tensor, value: Tensor) 表示将所有满足 func 的值都设置成 value, pytorch 惯例，方法尾部不带 _ 的一般是创建新的内存,不创建新的内存使用 tensor.masked_fill_(input: Tensor, mask: Tensor, value: Tensor)
- PyTorch 的自动求导机制（Autograd）需要保留前向传播时的某些张量状态来计算梯度。所以前向传播一般使用非原位操作
- 一般在不需要计算梯度的操作的时候使用 in-place 操作
- float('inf') 表示无穷
- 切记这里是上三角的位置设置为 -∞，seq_q < seq_k 的位置为 -∞
- 在最后一个维度进行 softmax

In [ ]:
def causal_attention(Q, K, V):
    # Q, K, V shape: (batch, seq_len, d)
    
    batch, seq_len, d = Q.shape
    
    # 1. 计算缩放点积注意力分数
    # 公式: scores = Q @ K.T / sqrt(d)
    # 使用 bmm (batch matrix multiplication) 处理批次
    scores = torch.bmm(Q, K.transpose(1, 2))  # (batch, seq_len, seq_len)
    scores = scores / math.sqrt(d)
    
    # 2. 应用因果掩码 (Causal Mask)
    # 创建上三角矩阵 (不包括对角线)，将未来位置的分数设为 -inf 切记这里是上三角的位置设置为 -∞，seq_q < seq_k 的值为 -∞
    # mask[i, j] = 1 当 j > i (未来位置)
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    # 扩展到批次维度 (B, seq_len, seq_len)
    mask = mask.unsqueeze(0).to(Q.device)
    # 将 Q <= K 的 seq 位置的分数设为 -inf，这样 softmax 后权重接近 0
    scores.masked_fill_(mask, -float('inf'))
    
    # 3. 应用 softmax 得到注意力权重
    # 在最后一个维度上进行 softmax
    attn_weights = torch.softmax(scores, dim=-1)
    
    # 4. 计算输出
    # 公式: out = attn_weights @ V
    out = torch.bmm(attn_weights, V)
    
    return out

In [ ]:
# 🧪 调试
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("输出形状:", out.shape)          # 应为 (1, 4, 8)
print("位置 0 是否等于 V[:,0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # 应该为 True

In [ ]:
from torch_judge import check
check('causal_attention')